In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from zoneinfo import ZoneInfo

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports loaded")


✅ Imports loaded


## Step 1: Load Recent Arb Data

Let's load the most recent arbitrage data to see which opportunities involved Bovada.


In [2]:
# Find most recent arb files
data_dir = Path('../data/04_output/nba/arbs')

if not data_dir.exists():
    print(f"❌ Data directory not found: {data_dir}")
else:
    arb_files = sorted(data_dir.glob('arb_output_*.csv'))
    print(f"✅ Found {len(arb_files)} arb files")
    
    if arb_files:
        # Load most recent 5 files
        recent_files = arb_files[-5:]
        print(f"\nMost recent files:")
        for f in recent_files:
            print(f"  - {f.name}")
        
        # Load and combine
        dfs = []
        for f in recent_files:
            df = pd.read_csv(f)
            df['source_file'] = f.name
            dfs.append(df)
        
        recent_arbs = pd.concat(dfs, ignore_index=True)
        print(f"\n✅ Loaded {len(recent_arbs)} rows from recent files")
        print(f"\nColumns: {recent_arbs.columns.tolist()}")
    else:
        print("❌ No arb files found")
        recent_arbs = None


✅ Found 2429 arb files

Most recent files:
  - arb_output_20251214_140108.csv
  - arb_output_20251214_140208.csv
  - arb_output_20251214_140307.csv
  - arb_output_20251214_140408.csv
  - arb_output_20251214_140508.csv

✅ Loaded 0 rows from recent files

Columns: ['player', 'market', 'line', 'best_over_odds', 'best_over_book', 'best_over_implied', 'best_under_odds', 'best_under_book', 'best_under_implied', 'total_prob', 'expected_profit_pct', 'is_arb', 'over_stake', 'under_stake', 'over_return', 'under_return', 'guaranteed_profit', 'total_wager', 'recommendation', 'game', 'game_time', 'num_bookmakers', 'source_file']


## Step 2: Analyze Bovada Involvement in Arbs


In [3]:
if recent_arbs is not None and len(recent_arbs) > 0:
    # Filter to actual arbs
    if 'is_arb' in recent_arbs.columns:
        arbs_only = recent_arbs[recent_arbs['is_arb'] == True].copy()
        print(f"Total arb opportunities: {len(arbs_only)}")
    else:
        arbs_only = recent_arbs.copy()
    
    # Check which involve Bovada
    if 'best_over_book' in arbs_only.columns and 'best_under_book' in arbs_only.columns:
        bovada_overs = (arbs_only['best_over_book'] == 'bovada').sum()
        bovada_unders = (arbs_only['best_under_book'] == 'bovada').sum()
        bovada_involved = ((arbs_only['best_over_book'] == 'bovada') | 
                          (arbs_only['best_under_book'] == 'bovada')).sum()
        
        print(f"\n📊 Bovada Involvement:")
        print(f"  Bovada best OVER odds: {bovada_overs} ({bovada_overs/len(arbs_only)*100:.1f}%)")
        print(f"  Bovada best UNDER odds: {bovada_unders} ({bovada_unders/len(arbs_only)*100:.1f}%)")
        print(f"  Bovada involved in arb: {bovada_involved} ({bovada_involved/len(arbs_only)*100:.1f}%)")
        
        # Sample some Bovada arbs
        print(f"\n🎰 Sample Bovada Arbs:")
        bovada_arbs = arbs_only[
            (arbs_only['best_over_book'] == 'bovada') | 
            (arbs_only['best_under_book'] == 'bovada')
        ].copy()
        
        if len(bovada_arbs) > 0:
            display_cols = ['player', 'market', 'line', 'best_over_book', 'best_over_odds', 
                          'best_under_book', 'best_under_odds', 'expected_profit_pct', 'source_file']
            display_cols = [c for c in display_cols if c in bovada_arbs.columns]
            print(bovada_arbs[display_cols].head(10).to_string(index=False))
else:
    print("❌ No data to analyze")


❌ No data to analyze


## Step 3: Fetch LIVE Data with Timestamps

Let's make a live API call to see current timestamp patterns for Bovada vs other bookmakers. **This will cost ~10-20 credits.**


In [4]:
import sys
sys.path.append('../api_setup')

from odds_api_efficient import OddsAPIEfficient

print("🔌 Connecting to The Odds API...\n")

try:
    api = OddsAPIEfficient()
    print("✅ API initialized")
    
    # Fetch live props
    print("\n📡 Fetching live NBA player props...")
    print("(This will cost ~10-20 credits)\n")
    
    props_data = api.get_nba_player_props(
        markets='player_points,player_threes,player_rebounds,player_assists',
        use_cache=False  # Force fresh data
    )
    
    if props_data:
        print(f"\n✅ Got data for {len(props_data)} games")
    else:
        print("\n❌ No data returned")
        props_data = None
        
except Exception as e:
    print(f"❌ Error: {e}")
    props_data = None


🔌 Connecting to The Odds API...

✅ API initialized

📡 Fetching live NBA player props...
(This will cost ~10-20 credits)

🏀 Fetching NBA player props: player_points,player_threes,player_rebounds,player_assists
📅 Time: 2025-12-14 13:07

❌ Error: 422 Client Error: Unprocessable Entity for url: https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?regions=us&markets=player_points%2Cplayer_threes%2Cplayer_rebounds%2Cplayer_assists&oddsFormat=american&apiKey=ef9e70e0037de7897681366406ca4461

❌ No data returned


## Step 4: Extract and Analyze Timestamps

Parse the raw API response to extract bookmaker-level timestamps.


In [5]:
if props_data is not None and len(props_data) > 0:
    # Extract timestamp data from all games and bookmakers
    timestamp_records = []
    
    api_fetch_time = datetime.now(ZoneInfo('UTC'))
    
    for game in props_data:
        game_id = game.get('id')
        game_info = f"{game.get('away_team')} @ {game.get('home_team')}"
        game_commence = game.get('commence_time')
        
        for bookmaker in game.get('bookmakers', []):
            book_key = bookmaker.get('key')
            book_title = bookmaker.get('title')
            book_last_update = bookmaker.get('last_update')
            
            # Get market-level timestamps too
            markets = bookmaker.get('markets', [])
            
            for market in markets:
                market_key = market.get('key')
                market_last_update = market.get('last_update')
                num_outcomes = len(market.get('outcomes', []))
                
                timestamp_records.append({
                    'game': game_info,
                    'game_commence': game_commence,
                    'bookmaker': book_title,
                    'bookmaker_key': book_key,
                    'market': market_key,
                    'bookmaker_last_update': book_last_update,
                    'market_last_update': market_last_update,
                    'num_outcomes': num_outcomes,
                    'api_fetch_time': api_fetch_time
                })
    
    ts_df = pd.DataFrame(timestamp_records)
    
    # Convert timestamps
    ts_df['bookmaker_last_update'] = pd.to_datetime(ts_df['bookmaker_last_update'], utc=True)
    ts_df['market_last_update'] = pd.to_datetime(ts_df['market_last_update'], utc=True)
    ts_df['game_commence'] = pd.to_datetime(ts_df['game_commence'], utc=True)
    
    # Calculate staleness (minutes since last update)
    ts_df['minutes_since_update'] = (ts_df['api_fetch_time'] - ts_df['market_last_update']).dt.total_seconds() / 60
    
    print(f"✅ Extracted {len(ts_df)} bookmaker/market combinations")
    print(f"\nAPI Fetch Time: {api_fetch_time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    print(f"Unique bookmakers: {ts_df['bookmaker'].nunique()}")
    print(f"Bookmakers: {sorted(ts_df['bookmaker'].unique())}")
else:
    print("❌ No props data to analyze")
    ts_df = None


❌ No props data to analyze
